# 05 Recommender — Graphic Novels

Goal:
Build a content-based recommendation system using TF-IDF and cosine similarity.

Input:
`../Data/Clean/merged_graphic_novels.csv`

Outputs:
- `../Models/graphic_vectorizer.pkl`
- `../Models/graphic_tfidf_matrix.npz`
- `../Models/graphic_books_index.json`

In [26]:
import pandas as pd
import pickle
import json
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy import sparse

In [27]:
df = pd.read_csv("../Data/Clean/merged_graphic_novels.csv")

df.shape

(351, 16)

In [28]:
df["combined_text"] = (
    df["clean_description"] + " " +
    df["clean_description"] + " " +  # repeat to weight it more
    df["title"] + " " +
    df["author"]
)

In [29]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=8000
)

tfidf_matrix = vectorizer.fit_transform(df["combined_text"])

tfidf_matrix.shape

(351, 7643)

In [30]:
# Build reccommendation function
def recommend_books(title, df, tfidf_matrix, top_n=5):
    title = title.lower()

    matches = df[df["title"].str.lower().str.contains(title, na=False)]

    if matches.empty:
        return pd.DataFrame(columns=["title", "author", "description", "cover_url"])

    idx = matches.index[0]

    cosine_scores = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()

    similar_indices = cosine_scores.argsort()[::-1][1:top_n + 1]

    recommendations = df.iloc[similar_indices][[
        "title",
        "author",
        "first_publish_year",
        "description",
        "cover_url"
    ]].copy()

    recommendations["similarity_score"] = cosine_scores[similar_indices]

    return recommendations

In [31]:
# Test the recommendation function
df[["title", "author"]].sample(10)

,title,author
50,Sobriety,"Daniel R. Maurer, Spencer Amundson"
261,David Hockney,"David Hockney, Sarah Howgate, Barbara Stern Sh..."
36,A Study in Scarlet [graphic novel],"Ian Eddington, Ian Edginton"
273,The Battle of the Alamo,"Matt Doeden, Charles Barnett III, Phil Miller"
311,La guerre d'Alan,Emmanuel Guibert
7,Gulliver's Travels,Jonathan Swift
184,Teleny and Camille,Jon Macy
23,Big Jim Believes,Dav Pilkey
8,Graphic Novels,Paul Gravett
326,The Story of Jamestown,"Eric Braun, Steve Erwin, Keith Williams, Charl..."


In [32]:
# Try one title from the dataset
recommend_books("maus", df, tfidf_matrix, top_n=5)

,title,author,first_publish_year,description,cover_url,similarity_score
328,The Bund,Sharon Rudahl,2023.0,"Told in an engaging graphic novel format, The ...",https://covers.openlibrary.org/b/id/14844753-M...,0.154244
304,Ghetto brother,Julian Voloj,2015.0,"""An engrossing and counter view of one of the ...",https://covers.openlibrary.org/b/id/11637611-M...,0.105840
73,"""How Come Boys Get to Keep Their Noses?""",Tahneer Oksman,2016.0,American comics reflect the distinct sensibili...,https://covers.openlibrary.org/b/id/7401506-M.jpg,0.099644
136,"Comic Books, Graphic Novels and the Holocaust",Claire Gorrara,NaN,This book analyses the portrayals of the Holoc...,NaN,0.095330
301,Displacement,Lucy Knisley,2015.0,"""In the next installment of her graphic memoir...",https://covers.openlibrary.org/b/id/12195671-M...,0.089332


In [21]:
# Create models folder
models_dir = Path("../Models")
models_dir.mkdir(parents=True, exist_ok=True)

In [22]:
# Save vectorizer
with open(models_dir / "graphic_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

In [23]:
# Save TF-IDF matrix
sparse.save_npz(models_dir / "graphic_tfidf_matrix.npz", tfidf_matrix)

In [24]:
# Save book index for later use in the API
books_index = df[[
    "title",
    "author",
    "first_publish_year",
    "description",
    "cover_url",
    "combined_text"
]].to_dict(orient="records")

with open(models_dir / "graphic_books_index.json", "w", encoding="utf-8") as f:
    json.dump(books_index, f, ensure_ascii=False, indent=2)

In [25]:
# Save the cleaned DataFrame to a new CSV file
df.to_csv("../Data/Clean/merged_graphic_novels.csv", index=False)

print("Graphic novels recommender files saved.")

Graphic novels recommender files saved.
